In [22]:
from __future__ import annotations

import numpy as np
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import normalize_embeddings
from typing import List, Tuple
import sys
from datetime import datetime

# ----------------------------------------------------------------------
# 1. TextSimilarity class
# ----------------------------------------------------------------------
class TextSimilarity:

    def __init__(
        self,
        model_name: str = "intfloat/e5-large-v2",   # change to e5-small-v2 for speed
        device: str | None = None,
        prefix: str = "query: ",
    ):
        print(f"Loading model '{model_name}' …", end="", flush=True)
        self.model = SentenceTransformer(model_name, device=device)
        self.prefix = prefix
        print(" Done")

    def _add_prefix(self, texts: List[str]) -> List[str]:
        return [f"{self.prefix}{t}" for t in texts]

    def embed(self, texts: List[str]) -> np.ndarray:
        prefixed = self._add_prefix(texts)
        return self.model.encode(
            prefixed,
            normalize_embeddings=True,
            show_progress_bar=False,
            batch_size=32,
        )

    def cosine(self, a: np.ndarray, b: np.ndarray) -> float:
        return float(np.dot(a, b))

    def similarity(self, ref: str, user: str) -> float:
        emb = self.embed([ref, user])
        return self.cosine(emb[0], emb[1])

    def batch_similarity(self, pairs: List[Tuple[str, str]]) -> List[float]:
        refs = [p[0] for p in pairs]
        users = [p[1] for p in pairs]
        emb = self.embed(refs + users)
        ref_emb = emb[: len(pairs)]
        user_emb = emb[len(pairs) :]
        return [float(np.dot(r, u)) for r, u in zip(ref_emb, user_emb)]


# ----------------------------------------------------------------------
# 2. Test data
# ----------------------------------------------------------------------
TEST_CASES = [
    # (ref, user, expected_min, name)
    ("I love coding.", "I love coding.", 0.99, "exact"),
    (
        "I am a computer science student with a passion for creating software.",
        "I study CS and enjoy building programs.",
        0.80,
        "synonyms",
    ),
    (
        "The quick brown fox jumps over the lazy dog.",
        "A fast brown fox leaps over a sleepy dog.",
        0.85,
        "rephrase",
    ),
    ("Python is a programming language.", "I like eating pizza.", 0.30, "unrelated"),
    ("HELLO world!!!", "hello, world.", 0.95, "case_punct"),
    ("  AI   is   fun  ", "AI is fun", 0.99, "whitespace"),
    (
        "میں کمپیوٹر سائنس کا طالب علم ہوں۔",
        "I am a computer science student.",
        0.70,
        "urdu_en",
    ),
    ("Meeting on 10 Nov 2025 at 6 PM.", "Conference scheduled for November 10, 2025, 18:00.", 0.85, "date"),
    ("I hate bugs in code.", "I love debugging software.", 0.45, "opposite"),
    ("Yes", "No", 0.20, "short"),
    (
        "def add(a, b): return a + b",
        "A function that returns the sum of two numbers.",
        0.75,
        "code_desc",
    ),
    (
        "Machine learning is a field of artificial intelligence that uses statistical techniques to give computer systems the ability to learn from data.",
        "ML is about learning from data.",
        0.78,
        "long_short",
    ),
]


# ----------------------------------------------------------------------
# 3. Test runner (pretty table, no external deps)
# ----------------------------------------------------------------------
def run_all_tests(model_name: str = "intfloat/e5-large-v2", device: str | None = None) -> None:
    sim = TextSimilarity(model_name=model_name, device=device)

    print("\n" + "=" * 90)
    print(f"Running {len(TEST_CASES)} tests with model: {model_name}")
    print("=" * 90)

    passed = 0
    tolerance = 0.03
    max_text_len = 80  # Truncate long texts for clean output

    for i, (ref, usr, exp_min, name) in enumerate(TEST_CASES, 1):
        score = sim.similarity(ref, usr)
        ok = score >= exp_min - tolerance
        status = "PASS" if ok else "FAIL"
        passed += ok

        # Truncate long texts
        ref_short = (ref[:max_text_len] + '...') if len(ref) > max_text_len else ref
        usr_short = (usr[:max_text_len] + '...') if len(usr) > max_text_len else usr

        print(f"{i:2d}. [{status}] {name:12s} | score = {score:.4f} (>= {exp_min:.2f}) expected")
        print(f"     Ref : {ref_short}")
        print(f"     User: {usr_short}")
        print("-" * 90)

    print("=" * 90)
    print(f"Summary: {passed}/{len(TEST_CASES)} tests passed")
    print(f"Time: {datetime.now().strftime('%Y-%m-%d %I:%M %p PKT')} | @wrhgye_bhai")
    print("=" * 90)


# ----------------------------------------------------------------------
# 4. Entry point – run when the file is executed directly
# ----------------------------------------------------------------------
if __name__ == "__main__":
    # Here we detect if we are in Colab and auto‑run.
    try:
        import google.colab  # type: ignore
        IN_COLAB = True
    except Exception:
        IN_COLAB = False

    if IN_COLAB:
        run_all_tests(model_name="intfloat/e5-large-v2", device=None)
    else:
        # Local run – you can pass args if you like
        run_all_tests()

Loading model 'intfloat/e5-large-v2' … Done

Running 12 tests with model: intfloat/e5-large-v2
 1. [PASS] exact        | score = 1.0000 (>= 0.99) expected
     Ref : I love coding.
     User: I love coding.
------------------------------------------------------------------------------------------
 2. [PASS] synonyms     | score = 0.9099 (>= 0.80) expected
     Ref : I am a computer science student with a passion for creating software.
     User: I study CS and enjoy building programs.
------------------------------------------------------------------------------------------
 3. [PASS] rephrase     | score = 0.9600 (>= 0.85) expected
     Ref : The quick brown fox jumps over the lazy dog.
     User: A fast brown fox leaps over a sleepy dog.
------------------------------------------------------------------------------------------
 4. [PASS] unrelated    | score = 0.7614 (>= 0.30) expected
     Ref : Python is a programming language.
     User: I like eating pizza.
----------------------